# 05 change detection

Continuous change only. Delta NDVI = post minus pre (negative decline); dNBR/nbr_loss = pre minus post (positive decline). SAR power ratios are dependent transforms and use the during-event date. No classes, ML or AOI-wide offset removal.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yaml').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT.parent / 'work' / 'mplconfig'))
from pipeline.config import load_config
from pipeline.forest_notebooks import preview
from IPython.display import display
cfg = load_config(ROOT / 'config.yaml')
SENSORS = list(cfg['forest_change']['enabled_sensors'])
# Edit config.yaml first. An in-session override may instead be made here.
# SENSORS = ['landsat', 'sentinel2', 'opera', 'hyp3']
OUT = cfg.path(cfg['forest_change']['outputs'])
print('Enabled sensors:', SENSORS)


In [ ]:
from pipeline.forest_stack import load_stack, optical_change, save_stack, stack_path
from pipeline.forest_sar import sar_change
for sensor in SENSORS:
    ds = load_stack(cfg,sensor)
    ds = optical_change(ds) if sensor in ['landsat','sentinel2'] else sar_change(cfg,ds)
    save_stack(ds,stack_path(cfg,sensor))
    display(ds)
    metric = 'delta_ndvi' if sensor in ['landsat','sentinel2'] else 'vv_log_ratio_db'
    display(ds[metric].to_series().describe())